# Experiment 1 — Abundance-weighted Musanga satellite regression

**Standalone, CPU-only, cached; no Earth Engine and no GEDI extraction.**

Compare the **exact existing 50 m S2 + 64-dimensional Earth embedding HGB baseline** with two predeclared, train-only abundance-weighting schemes, retaining four strict leave-one-forest-out (LOFO) splits, identical balanced draw per ensemble member, five ensemble members, and the unchanged HGB hyperparameters.

Training bins: 0–10%, 10–20%, 20–40%, 40–60%, ≥60% (left-closed). Per sampled training set: each row in bin *b* gets weight proportional to `(N / (K × n_b))^α`; `α = 0` is baseline, `0.5` softened inverse frequency, and `1.0` stronger inverse frequency. Weights are clipped to [0.25, 4], normalized to mean one, and computed **only from training cells**. No tuning using held-out labels.

Uses the original NB06 `draw_balanced` algorithm and the same `HistGradientBoostingRegressor` settings. **Retrains the baseline on the same selections** for a paired, interpretable comparison; does not overwrite previous predictions.

Reports forest-balanced global R²/MAE/RMSE/bias, individual forest scores, MOS reference ≥40% recall and bias, and additional ≥60% sensitivity. Files are resumable per held-out forest, weighting scheme and member. **No winner is selected by optimizing on the four held-out forests**; outputs are comparisons, not new unbiased hyperparameter-tuning results.

The 14,454-cell frozen set differs from the earlier 14,023-cell Planet-matched comparison; **don't compare metrics across those different evaluation sets as if paired**.

In [ ]:
# EXPERIMENT 1 — frozen 50m S2+EMB; 4 strict LOFO forests; 3 training-only weight schemes.
from pathlib import Path
import os,json,hashlib,time
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from threadpoolctl import threadpool_limits
from IPython.display import display
threadpool_limits(1)
ROOTS=[
 Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06'),
 Path('/content/drive/MyDrive/Musanga/Final/Outputs/V3_1/06_spatial_planet_embeddings/CLEAN_NB06')]
if not any((r/'frozen_50m_reference.parquet').is_file() for r in ROOTS):
    from google.colab import drive
    drive.mount('/content/drive')
found=[r for r in ROOTS if (r/'frozen_50m_reference.parquet').is_file()]
assert len(found)==1,('Expected one frozen Musanga reference file',found)
ROOT=found[0]
D=pd.read_parquet(ROOT/'frozen_50m_reference.parquet')
SITES=('Baego','MOS','Yangambi','Yoko')
KEYS=['site','mosaic','row','col']
assert len(D)==14454 and not D.duplicated(KEYS).any()
assert D.groupby('site').size().to_dict()=={'Baego':480,'MOS':11494,'Yangambi':1383,'Yoko':1097}
cols_s2=[c for c in D if c.startswith(('RAW_ANNUAL__','S2_QUARTERLY_3YR__'))]
cols_emb=[c for c in D if c.startswith('EMB_ANNUAL__')]
assert len(cols_s2)>0 and len(cols_emb)==64
features=cols_s2+cols_emb
assert np.isfinite(D[features+['target_fraction','x','y']].to_numpy(float)).all()
assert D.target_fraction.between(0,1).all()
X=D[features].to_numpy(np.float32)
y=D.target_fraction.to_numpy(float)
forest=D.site.to_numpy()
xy=D[['x','y']].to_numpy(float)
SEED=20261026;CAP=1500;MEMBERS=5;HIGH=.40
PARAM=dict(learning_rate=.06,max_iter=180,max_leaf_nodes=20,
           min_samples_leaf=35,l2_regularization=2,early_stopping=False)
# Fixed, NON-TUNED schemes. Alpha=0 is the previously used uniform sample weights.
SCHEMES={'baseline':0.,'soft_sqrt_inverse':.5,'inverse_frequency':1.}
WEIGHT_EDGES=np.array([0.,.10,.20,.40,.60,1.0000001])
OUT=ROOT/'09_abundance_weighted_LOFO';OUT.mkdir(parents=True,exist_ok=True)
source_hash=hashlib.sha256(pd.util.hash_pandas_object(
 D[KEYS+['x','y','target_fraction']+features],index=False).values.tobytes()).hexdigest()
cfg=dict(version='EXP1_S2_EMB_HGB_LOFO_weights_v1',
         seed=SEED,cap=CAP,members=MEMBERS,model=PARAM,
         forest_order=list(SITES),schemes=SCHEMES,
         bin_edges=WEIGHT_EDGES.tolist(),clip=[.25,4.],
         source_sha256=source_hash,n_cells=len(D),n_features=len(features))
cfgfile=OUT/'config.json'
if cfgfile.is_file():assert json.loads(cfgfile.read_text())==cfg,'Stored experiment config differs'
else:cfgfile.write_text(json.dumps(cfg,indent=2))
# EXACT same balanced draw routine as NB06 block 01A, reused for every scheme.
def draw_balanced(pool,seed):
    rng=np.random.default_rng(seed);selected=[]
    for name in SITES:
        ix=pool[forest[pool]==name]
        if not len(ix):continue
        cap=min(CAP,len(ix))
        if len(ix)<=cap:selected.extend(ix.tolist());continue
        labels=np.digitize(y[ix],[.10,.20,.40])
        bins={b:ix[labels==b] for b in range(4)}
        n=min(len(bins[3]),int(np.ceil(cap*.25)))
        pick=rng.choice(bins[3],n,replace=False).tolist() if n else []
        bins[3]=np.setdiff1d(bins[3],pick)
        while len(pick)<cap:
            active=[b for b in bins if len(bins[b])]
            assert active
            for b in active:
                n=min(len(bins[b]),max(1,(cap-len(pick))//len(active)),cap-len(pick))
                if n:
                    take=rng.choice(bins[b],n,replace=False)
                    pick.extend(take.tolist());bins[b]=np.setdiff1d(bins[b],take)
                if len(pick)==cap:break
        selected.extend(pick)
    return np.array(sorted(selected),dtype=int)
def training_weights(train_ids,alpha):
    if alpha==0:return np.ones(len(train_ids),float)
    label=np.searchsorted(WEIGHT_EDGES[1:-1],y[train_ids],side='right')
    counts=np.bincount(label,minlength=5)
    active=counts>0
    # Overall inverse frequency across the balanced training DRAW. Weights
    # do not inspect any held-out test labels or test forest frequencies.
    per_bin=np.ones(5)
    per_bin[active]=(len(label)/(active.sum()*counts[active]))**alpha
    w=np.clip(per_bin[label],.25,4.)
    return w/w.mean()
def score(actual,pred):
    err=pred-actual
    den=np.mean((actual-actual.mean())**2)
    for_high=actual>=HIGH
    for_very=actual>=.60
    return dict(n=len(actual),mae_pp=100*np.mean(np.abs(err)),
      rmse_pp=100*np.sqrt(np.mean(err**2)),bias_pp=100*np.mean(err),
      mse=float(np.mean(err**2)),target_mean=float(actual.mean()),
      target_var=float(den),r2=1-np.mean(err**2)/den if den>0 else np.nan,
      high_n=int(for_high.sum()),
      high_bias_pp=100*np.mean(err[for_high]) if for_high.any() else np.nan,
      high_recall=float(np.mean(pred[for_high]>=HIGH)) if for_high.any() else np.nan,
      very_high_n=int(for_very.sum()),
      very_high_bias_pp=100*np.mean(err[for_very]) if for_very.any() else np.nan,
      very_high_recall=float(np.mean(pred[for_very]>=.60)) if for_very.any() else np.nan)
print('Exact frozen study counts:');display(D.groupby('site').size().rename('cells'))
print('Training recipe: S2+EMB, 5 members, fixed LOFO, HGB; model will fit 4 × 3 × 5 = 60 times.')
# Cache member predictions. Each member uses the SAME draw for all schemes.
train_rows=[]
for held in SITES:
    pool=np.flatnonzero(forest!=held)
    test=np.flatnonzero(forest==held)
    assert not (forest[pool]==held).any() and len(test)>0
    for member in range(MEMBERS):
        # Use original NB06 seed formula for feature family 'S2_EMB'.
        selected=draw_balanced(pool,SEED+107*member+sum(map(ord,held))+len('S2_EMB')*13)
        assert not np.intersect1d(selected,test).size
        b=np.searchsorted(WEIGHT_EDGES[1:-1],y[selected],side='right')
        counts=np.bincount(b,minlength=5)
        train_rows.append(dict(holdout=held,member=member,n_train=len(selected),
          n_train_high=int((y[selected]>=.4).sum()),
          n_train_very_high=int((y[selected]>=.6).sum()),
          **{f'bin_{j}_n':int(counts[j]) for j in range(5)}))
        for name,alpha in SCHEMES.items():
            cache=OUT/f'LOFO_{held}_{name}_m{member}.npz'
            if cache.is_file():
                with np.load(cache,allow_pickle=False) as z:
                    assert np.array_equal(z['test_index'],test)
                    assert len(z['pred'])==len(test)
                print('CACHE',held,name,member,flush=True)
                continue
            weights=training_weights(selected,alpha)
            assert np.isfinite(weights).all() and np.all(weights>0)
            start=time.monotonic()
            model=HistGradientBoostingRegressor(random_state=SEED+member,**PARAM)
            model.fit(X[selected],y[selected],sample_weight=weights)
            prediction=np.clip(model.predict(X[test]),0,1)
            temporary=cache.with_suffix('.partial')
            with open(temporary,'wb') as fp:
                np.savez_compressed(fp,test_index=test,pred=prediction.astype(np.float32))
            os.replace(temporary,cache)
            print(f'FIT {held:8s} {name:23s} member {member} | '
                  f'train={len(selected)}, test={len(test)}, '
                  f'{time.monotonic()-start:.1f}s',flush=True)
pd.DataFrame(train_rows).to_csv(OUT/'training_abundance_audit.csv',index=False)
# Every scheme must have 20 complete member caches before scoring.
all_preds={}
for name in SCHEMES:
    pred=np.full(len(D),np.nan,dtype=float)
    for held in SITES:
        test=np.flatnonzero(forest==held)
        fitted=[]
        for member in range(MEMBERS):
            f=OUT/f'LOFO_{held}_{name}_m{member}.npz'
            assert f.is_file(),('Incomplete fit',f)
            with np.load(f,allow_pickle=False) as z:
                assert np.array_equal(z['test_index'],test)
                fitted.append(z['pred'].astype(float))
        assert np.isnan(pred[test]).all()
        pred[test]=np.mean(fitted,axis=0)
    assert np.isfinite(pred).all()
    all_preds[name]=pred
rows=[]
for name,pred in all_preds.items():
    for site in SITES:
        mask=forest==site
        rows.append(dict(scheme=name,forest=site,**score(y[mask],pred[mask])))
per=pd.DataFrame(rows)
global_rows=[]
for name,subset in per.groupby('scheme',sort=False):
    mu=subset.target_mean.mean()
    v=np.mean(subset.target_var+(subset.target_mean-mu)**2)
    mos=subset[subset.forest=='MOS'].iloc[0]
    global_rows.append(dict(scheme=name,n_total=len(D),n_forests=4,
      equal_forest_R2=float(1-subset.mse.mean()/v),
      equal_forest_MAE_pp=float(subset.mae_pp.mean()),
      equal_forest_RMSE_pp=float(100*np.sqrt(subset.mse.mean())),
      equal_forest_bias_pp=float(subset.bias_pp.mean()),
      MOS_MAE_pp=float(mos.mae_pp),
      MOS_high_n=int(mos.high_n),
      MOS_high_bias_pp=float(mos.high_bias_pp),
      MOS_high_recall=float(mos.high_recall),
      MOS_very_high_n=int(mos.very_high_n),
      MOS_very_high_bias_pp=float(mos.very_high_bias_pp),
      MOS_very_high_recall=float(mos.very_high_recall)))
glob=pd.DataFrame(global_rows)
baseline=glob.loc[glob.scheme=='baseline'].iloc[0]
for col in ['equal_forest_R2','equal_forest_MAE_pp','MOS_MAE_pp',
            'MOS_high_bias_pp','MOS_high_recall']:
    glob['change_vs_baseline__'+col]=glob[col]-baseline[col]
print('PRIMARY: same 14,454 held-out cells; equal forest weight, 4 strict LOFO folds')
display(glob.round(3))
print('PER FOREST (all 3 weighting schemes):')
display(per[['scheme','forest','n','r2','mae_pp','bias_pp',
             'high_n','high_bias_pp','high_recall',
             'very_high_n','very_high_bias_pp','very_high_recall']].round(3))
glob.to_csv(OUT/'EXP1_equal_forest_LOFO_summary.csv',index=False)
per.to_csv(OUT/'EXP1_per_forest_LOFO.csv',index=False)
pd.DataFrame({**{k:D[k] for k in KEYS},'target_fraction':y,
              **{'pred_'+name:pred for name,pred in all_preds.items()}}
             ).to_parquet(OUT/'EXP1_paired_LOFO_predictions.parquet',index=False)
# Per-forest balanced plot; show MOS high-cover region explicitly.
fig,ax=plt.subplots(2,2,figsize=(11,10),layout='constrained')
for a,site in zip(ax.flat,SITES):
    ids=np.flatnonzero(forest==site)
    rng=np.random.default_rng(SEED)
    # Deterministic visual subsample for large MOS; metrics always use ALL cells.
    if len(ids)>1300: ids=np.sort(rng.choice(ids,1300,replace=False))
    a.plot([0,100],[0,100],ls='--',lw=1,c='black')
    for name,col in [('baseline','#53667e'),('soft_sqrt_inverse','#15827d'),
                     ('inverse_frequency','#c57a28')]:
        a.scatter(y[ids]*100,all_preds[name][ids]*100,label=name,
                  alpha=.3,s=9,c=col,rasterized=True)
    a.axvline(40,c='grey',alpha=.6,ls=':')
    a.set(xlim=(-2,102),ylim=(-2,102),xlabel='UAV Musanga cover (%)',
          ylabel='Held-out S2 + EMB prediction (%)',title=site)
    a.grid(alpha=.1)
ax.flat[0].legend(fontsize=8)
fig.savefig(OUT/'EXP1_LOFO_scatter_per_forest.png',dpi=190)
plt.show();plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for a,col,title in [(axes[0],'MOS_high_bias_pp','MOS bias for reference >=40% (pp)'),
                    (axes[1],'equal_forest_MAE_pp','Equal-forest MAE (pp)')]:
    a.bar(glob.scheme,glob[col])
    a.set(title=title,ylabel='Percentage points')
    a.tick_params(axis='x',labelrotation=15)
fig.savefig(OUT/'EXP1_global_and_MOS_comparison.png',dpi=190)
plt.show();plt.close(fig)
print('COMPLETE:',OUT)
print('Note: 14,454-cell frozen baseline; do not equate with 14,023-cell Planet-matched baseline.')
